# Notebook 1: Create Provider and Patient tables
Creates ontology-aligned `provider` and `patient` Delta tables and loads demo data.

In [ ]:
import random
from datetime import date, timedelta
from pyspark.sql import functions as F

SEED = 20260401
random.seed(SEED)

print(f'Using seed: {SEED}')

In [ ]:
# Provider columns (name must be first): name, providerId, specialty, licenseNumber, department
specialties = [
    'Cardiology', 'Orthopedics', 'Pediatrics', 'Dermatology', 'Neurology',
    'Endocrinology', 'Oncology', 'Urology', 'Psychiatry', 'Pulmonology'
]
departments = [
    'Primary Care', 'Outpatient', 'Emergency', 'Inpatient', 'Specialty Clinic'
]
first_names = [
    'Alex', 'Jordan', 'Taylor', 'Morgan', 'Casey', 'Jamie', 'Avery', 'Riley', 'Quinn', 'Cameron',
    'Drew', 'Hayden', 'Parker', 'Reese', 'Skyler', 'Rowan', 'Elliot', 'Logan', 'Finley', 'Emerson'
]
last_names = [
    'Brooks', 'Carter', 'Hayes', 'Foster', 'Reed', 'Price', 'Bennett', 'Perry', 'Cooper', 'Morris',
    'Watson', 'Bailey', 'Murphy', 'Griffin', 'Ross', 'Ward', 'Kelly', 'Bell', 'Myers', 'Powell'
]

providers = []
for i in range(20):
    providers.append({
        'name': f"Dr. {first_names[i]} {last_names[i]}",
        'providerId': f"PRV{i+1:03d}",
        'specialty': specialties[i % len(specialties)],
        'licenseNumber': f"LIC{300000 + i}",
        'department': departments[i % len(departments)]
    })

providers_df = spark.createDataFrame(providers)
providers_df = providers_df.select('name', 'providerId', 'specialty', 'licenseNumber', 'department')
providers_df.write.mode('overwrite').format('delta').saveAsTable('provider')

display(spark.table('provider').orderBy('providerId'))

In [ ]:
# Patient columns (name must be first): name, patientId, mrn, dateOfBirth, bloodType, allergies
patient_first = [
    'Liam', 'Olivia', 'Noah', 'Emma', 'Elijah', 'Sophia', 'Mateo', 'Amelia', 'Lucas', 'Mia'
]
patient_last = [
    'Nguyen', 'Garcia', 'Patel', 'Smith', 'Johnson', 'Lopez', 'Brown', 'Davis', 'Wilson', 'Clark'
]
blood_types = ['A+', 'A-', 'B+', 'B-', 'AB+', 'AB-', 'O+', 'O-']
allergy_pool = ['None', 'Penicillin', 'Peanuts', 'Shellfish', 'Latex', 'Pollen']

patients = []
start_dob = date(1945, 1, 1)
end_dob = date(2016, 12, 31)
dob_span = (end_dob - start_dob).days

for i in range(50):
    fn = patient_first[i % len(patient_first)]
    ln = patient_last[(i * 3) % len(patient_last)]
    dob = start_dob + timedelta(days=random.randint(0, dob_span))
    patients.append({
        'name': f"{fn} {ln} {i+1}",
        'patientId': f"PAT{i+1:03d}",
        'mrn': f"MRN{700000 + i}",
        'dateOfBirth': dob.isoformat(),
        'bloodType': random.choice(blood_types),
        'allergies': random.choice(allergy_pool)
    })

patients_df = spark.createDataFrame(patients)
patients_df = patients_df.select('name', 'patientId', 'mrn', 'dateOfBirth', 'bloodType', 'allergies')
patients_df = patients_df.withColumn('dateOfBirth', F.to_date('dateOfBirth'))
patients_df.write.mode('overwrite').format('delta').saveAsTable('patient')

display(spark.table('patient').orderBy('patientId'))

In [ ]:
# Basic data quality checks for Notebook 1 outputs.
checks = {
    'provider_count_20': spark.table('provider').count() == 20,
    'patient_count_50': spark.table('patient').count() == 50,
    'provider_id_unique': spark.table('provider').select('providerId').distinct().count() == 20,
    'patient_id_unique': spark.table('patient').select('patientId').distinct().count() == 50
}

for k, v in checks.items():
    print(f'{k}: {v}')